# `sparsegf2.circuits.runner` - execute and analyze a realization

`simulate` is the one-call interface; `SimulationRunner` reuses the Clifford
table and lookup tables across many sample seeds. The runner executes each
`CircuitLayer`, computes the picture's final observables, and returns one
`SampleRecord`.

The current runner also supports named/custom online analyses, final-tableau
saving, depth checkpoints, live checkpoint callbacks, and checkpoint-granular
early stopping through `CHECKPOINT_STOP`.


In [1]:
from sparsegf2.circuits import CircuitConfig, simulate

cfg = CircuitConfig(
    graph_spec='cycle', n=8, picture='purification', p=0.2,
    gating_mode='random_pool', total_layers_override=6, base_seed=23,
)
record = simulate(
    cfg, sample_seed=5, analyses=['code_dimension', 'half_cut_entropy']
)
print('layers / gates / measurements:', record.total_layers, record.total_gates, record.total_measurements)
print('code dimension / half cut:', record.code_dimension, record.entropy_half_cut)
print('online analyses:', record.analyses)


layers / gates / measurements: 6 24 8
code dimension / half cut: 2 1
online analyses: {'code_dimension': 2, 'half_cut_entropy': 1}


## Schema-v2 RNG streams

Three streams have distinct SeedSequence entropy vectors:

- construction: `[base_seed, sample_seed]`;
- measurement outcomes: `[base_seed, sample_seed, 0x6D656173]` (`meas`);
- optional global scramble: `[base_seed, sample_seed, 0x73637262]` (`scrb`).

The tags make the outcome and scramble generators independent of circuit
construction and of each other. Distinct `(base_seed, sample_seed)` pairs never
collapse to the same scalar-sum seed. The scramble has its own stream, so
toggling it leaves the complete scheduled layer sequence unchanged.


In [2]:
import numpy as np
from sparsegf2.circuits import CircuitBuilder

base, sample = 23, 5
streams = {
    'construction': [base, sample],
    'measurement': [base, sample, 0x6D656173],
    'scramble': [base, sample, 0x73637262],
}
for name, entropy in streams.items():
    print(name, np.random.default_rng(entropy).integers(2**32, size=3))

plain = CircuitConfig(
    graph_spec='cycle', n=8, p=0.3, base_seed=base,
    total_layers_override=3, scramble=False,
)
scrambled = CircuitConfig(
    graph_spec='cycle', n=8, p=0.3, base_seed=base,
    total_layers_override=3, scramble=True,
)
a = CircuitBuilder(plain, sample).schedule()
b = CircuitBuilder(scrambled, sample).schedule()
same_schedule = all(
    x.gate_pairs == y.gate_pairs
    and np.array_equal(x.cliff_indices, y.cliff_indices)
    and x.meas_qubits == y.meas_qubits
    for x, y in zip(a, b, strict=True)
)
print('scramble toggle preserves schedule:', same_schedule)


construction [ 954131631 2235440846  589002896]
measurement [ 194845762 2587494093 3461585831]
scramble [1659694827 1092183887  102820310]
scramble toggle preserves schedule: True


## Tableau checkpoints

`checkpoint_layers` uses 1-based measured depth and records the state after
that layer's gates and measurements. With no callback, the record stores full
symplectic tableaux. Out-of-range indices are ignored. A checkpoint at the
executed final layer equals `final_tableau` when both are requested.


In [3]:
snapshot = simulate(
    cfg, sample_seed=7, save_tableau=True, checkpoint_layers=[2, 6, 99]
)
print('stored checkpoint layers:', sorted(snapshot.checkpoint_tableaux))
print(
    'final checkpoint equals final tableau:',
    np.array_equal(snapshot.checkpoint_tableaux[6], snapshot.final_tableau),
)


stored checkpoint layers: [2, 6]
final checkpoint equals final tableau: True


## Live callbacks and `CHECKPOINT_STOP`

A read-only callback computes an observable on the live tableau without saving
the tableau itself. Its return values populate `checkpoint_values`. Returning
`CHECKPOINT_STOP` stops after the current checkpoint; the sentinel is checked
before storage, so it never appears as a value. Final observables and optional
online analyses are still computed on the actual stopping state.


In [4]:
from sparsegf2 import code_dimension
from sparsegf2.circuits import CHECKPOINT_STOP

def read_k(sim, spec, layer):
    return int(code_dimension(sim, spec.n_system))

values = simulate(
    cfg, sample_seed=8, checkpoint_layers=[1, 3, 6],
    checkpoint_callback=read_k,
)
print('k at checkpoints:', values.checkpoint_values)

def stop_at_three(sim, spec, layer):
    if layer == 3:
        return CHECKPOINT_STOP
    return int(code_dimension(sim, spec.n_system))

early = simulate(
    cfg, sample_seed=9, checkpoint_layers=[1, 3, 6],
    checkpoint_callback=stop_at_three,
)
print('early-stop layer:', early.total_layers)
print('stored values:', early.checkpoint_values)
assert early.total_layers == 3 and 3 not in early.checkpoint_values


k at checkpoints: {1: 7, 3: 6, 6: 1}
early-stop layer: 3
stored values: {1: 7}


## Summary

The runner combines schema-v2 construction, independently tagged outcome and
scramble streams, exact final observables, implemented online analyses, and
nonperturbing checkpoint diagnostics. `CHECKPOINT_STOP` provides cheap
checkpoint-granular stopping while preserving a correct final record.
